# SECTION 1: RESLSTM MODEL TRAINING

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Reshape, LSTM, Dense, Dropout, BatchNormalization, ReLU, Add
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD, Adamax
from tensorflow.keras import backend as K
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import ModelCheckpoint

# ========== SETUP ==========
SAMPLE_RATE = 22050
BATCH_SIZE = 64
NUM_CLASSES = 5

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


# === Custom Attention Layer ===
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros")
        super().build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

# === ResLSTM WITH ATTENTION ===
def residual_block(x, filters, stride):
    shortcut = Conv2D(filters, kernel_size=(1, 1), strides=stride, padding='same')(x)
    shortcut = BatchNormalization()(shortcut)

    x = Conv2D(filters, kernel_size=(3, 3), strides=stride, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([shortcut, x])
    x = ReLU()(x)
    return x

def build_reslstm_attention(input_shape, num_classes):
    inputs = Input(shape=input_shape)

    x = Conv2D(16, kernel_size=(5, 5), strides=1, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)

    for filters, stride in zip([32, 64, 64], [2, 1, 1]):
        x = residual_block(x, filters, stride)
        x = Dropout(0.3)(x)

    x = Reshape((x.shape[1], x.shape[2]*x.shape[3]))(x)
    x = LSTM(32, return_sequences=True)(x)
    x = Dropout(0.3)(x)
    x = AttentionLayer()(x)

    output = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, output)

# === Universal Run Function ResLSTM ===
def run_attention_model(model_type="reslstm", optimizer_name="Adam", lr=0.001, epochs=60, split_ratio=(0.8, 0.1, 0.1), batch_size=32):
    X = np.load("X_hybrid_balanced.npy")
    y = np.load("y_hybrid_balanced.npy")
    X = X / np.max(X)
    X = X[..., np.newaxis]

    class_names = ["belly pain", "burping", "discomfort", "hungry", "tired"]
    num_classes = len(np.unique(y))
    y_cat = to_categorical(y, num_classes)

    X_temp, X_test, y_temp, y_test = train_test_split(X, y_cat, test_size=split_ratio[2], stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=split_ratio[1]/(split_ratio[0]+split_ratio[1]), stratify=y_temp.argmax(1), random_state=42)

    if model_type == "reslstm":
        model = build_reslstm_attention(X_train.shape[1:], num_classes)
    else:
        model = build_cnn_lstm_attention(X_train.shape[1:], num_classes)

    if optimizer_name == "SGD":
        optimizer = SGD(learning_rate=lr)
    elif optimizer_name == "Adamax":
        optimizer = Adamax(learning_rate=lr)
    elif optimizer_name == "RMSprop":
        optimizer = RMSprop(learning_rate=lr)
    else:
        optimizer = Adam(learning_rate=lr)

    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    checkpoint = ModelCheckpoint(
        filepath='best_model_ResLSTM.h5' if model_type == 'reslstm' else 'best_model_CNNLSTM.h5',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    )

    callbacks = [early_stop, checkpoint]

    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=epochs, batch_size=batch_size, verbose=1,
                        callbacks=callbacks)

    # === Final Evaluation
    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    print(f"\n✅ Final Training Accuracy: {train_acc * 100:.2f}%")
    print(f"✅ Final Validation Accuracy: {val_acc * 100:.2f}%")
    print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

    # === Report
    y_pred = model.predict(X_test).argmax(axis=1)
    y_true = y_test.argmax(axis=1)

    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap='Blues')
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.show()

    # === Accuracy Plot
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # === Loss Plot
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return train_acc, val_acc, test_acc



In [3]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.001 | 60 Epochs | Split 80:10:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.001,
    epochs=60,
    split_ratio=(0.8, 0.1, 0.1),
    batch_size=32
)



🔬 Running: ResLSTM | Adam | 0.001 | 60 Epochs | Split 80:10:10
Epoch 1/60
117/397 [=======>......................] - ETA: 11:00 - loss: 1.6049 - accuracy: 0.2358

ResourceExhaustedError: Graph execution error:

Detected at node 'gradient_tape/model/conv2d_9/Conv2D/Conv2DBackpropFilter' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "C:\Users\user\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
      app.start()
    File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
      self.io_loop.start()
    File "C:\Users\user\anaconda3\Lib\site-packages\tornado\platform\asyncio.py", line 195, in start
      self.asyncio_loop.run_forever()
    File "C:\Users\user\anaconda3\Lib\asyncio\windows_events.py", line 321, in run_forever
      super().run_forever()
    File "C:\Users\user\anaconda3\Lib\asyncio\base_events.py", line 607, in run_forever


In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.001 | 60 Epochs | Split 70:15:15")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.001,
    epochs=60,
    split_ratio=(0.7, 0.15, 0.15),
    batch_size=32
)


In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.001 | 60 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.001,
    epochs=60,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)


In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | SGD | 0.01 | 60 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="SGD", 
    lr=0.001,
    epochs=60,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)


In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | RMSprop | 0.001 | 60 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="RMSprop", 
    lr=0.001,
    epochs=60,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)

In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.0005 | 60 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.0005,
    epochs=60,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)

In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.0001 | 60 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.0001,
    epochs=60,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)

In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.001 | 40 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.001,
    epochs=40,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)

In [ ]:
print("\n==========================")
print("🔬 Running: ResLSTM | Adam | 0.001 | 20 Epochs | Split 70:20:10")
print("==========================")
run_attention_model(
    model_type="reslstm",
    optimizer_name="Adam", 
    lr=0.001,
    epochs=20,
    split_ratio=(0.7, 0.2, 0.1),
    batch_size=32
)

In [ ]:

from copy import deepcopy
import os
import pandas as pd
import json

best_val_acc = 0.0
best_model_state = None
best_optimizer_name = ""
results_list = []


In [ ]:
# Insert this block inside your training loop after validation

# === Check and Save Best Model ===
if val_acc > best_val_acc:
    best_val_acc = val_acc
    best_model_state = deepcopy(model.state_dict())
    best_optimizer_name = optimizer_name

# === Store results for comparison ===
results_list.append({
    "model": "ResLSTM",
    "optimizer": optimizer_name,
    "learning_rate": lr,
    "epochs": epochs,
    "train_accuracy": train_acc,
    "val_accuracy": val_acc,
    "precision": precision,
    "recall": recall,
    "f1_score": f1
})
